# Clase 24 — Laboratorios de MLOps

## Antes de empezar

Desde la raíz del repositorio del curso, ejecuta `git status` para comprobar la rama y revisar si tienes cambios locales. Desde `main`, ejecuta `git pull` para actualizar el material del curso. Si hay cambios locales que necesitas conservar, revísalos antes de actualizar.

**Resultado esperado:** implementar y verificar una capacidad operativa en el sistema de predicción de Green Taxi.

## Punto de partida

Green Taxi estima la **duración de un viaje antes de iniciarlo**. La aplicación recoge distancia aproximada, número de pasajeros, hora de recolección y zonas de origen y destino. FastAPI valida la entrada, utiliza el modelo cargado y devuelve los minutos estimados y la versión. La duración observada se obtiene de los registros históricos después del viaje.

En el [pipeline de Prefect](../modulo-02-ciclo-mlops/clase-13-pipelines.ipynb) orquestaste preparación, entrenamiento y evaluación. En [Model as a Service](../modulo-02-ciclo-mlops/clase-14-model-service-streamlit.ipynb) conectaste el modelo con la API y Streamlit. En [EC2](clase-20-aws-ec2-compose.ipynb) desplegaste esa aplicación con Docker Compose.

Una vez que el servicio responde, aparecen otras preguntas. ¿Cuánto tarda una predicción? ¿Cómo compruebas un cambio antes de integrarlo? ¿Qué modelo responde después de actualizar? ¿Cuándo conviene evaluar otro entrenamiento? ¿Qué aportó cada variable a una estimación?

## Capacidades de operación y madurez

En la [Clase 8](../modulo-02-ciclo-mlops/clase-08-introduccion-mlops.ipynb) usaste los niveles de madurez para identificar qué partes del sistema ya estaban automatizadas. Una prueba que corre en cada PR automatiza una comprobación de la aplicación. Desplegar un modelo aprobado requiere, además, identificar su versión y poder recuperar la anterior. Cuando el sistema observa sus entradas, una señal de cambio puede iniciar un nuevo ciclo de entrenamiento y evaluación.

| Capacidad | Qué compruebas en este sistema |
|---|---|
| Rendimiento | Cuánto tarda en responder la API y qué error tiene el modelo en datos reservados. |
| Integración continua (CI) | Si el cambio del PR conserva el contrato HTTP y la respuesta de predicción. |
| Despliegue continuo (CD) | Si una revisión integrada llega a EC2 y sirve la versión esperada. |
| Drift y reentrenamiento | Si cambió la distribución de las entradas y cómo se desempeña un candidato en viajes con duración observada. |
| Explicabilidad | Cómo contribuye cada entrada a una estimación particular y qué variables tienen mayor peso global según el método utilizado. |

Una capacidad aporta evidencia para diagnosticar la madurez. El nivel del sistema completo depende de cómo se integran y verifican sus capacidades.

## Ejemplo — Comparar dos modelos antes de desplegar

El **RMSE** resume el error de las predicciones en minutos y da más peso a errores grandes. La **mediana de la latencia HTTP** es el tiempo central de una serie de solicitudes. Aquí se calcula con cinco mediciones ficticias de cada versión, en milisegundos.

En este ejemplo, el equipo fija un límite de 200 ms de mediana. Entre las versiones que cumplen el límite, elige la de menor RMSE. En un laboratorio real, ambas versiones se prueban con las mismas entradas y el límite responde a las necesidades del servicio.

In [ ]:
from statistics import median

versiones = [
    {"version": "v1", "rmse_min": 4.8, "latencias_ms": [150, 160, 165, 170, 176]},
    {"version": "v2", "rmse_min": 4.2, "latencias_ms": [195, 201, 210, 216, 220]},
]
limite_ms = 200

aptas = []
for version in versiones:
    latencia_ms = median(version["latencias_ms"])
    print(version["version"], "RMSE:", version["rmse_min"], "min; mediana:", latencia_ms, "ms")
    if latencia_ms <= limite_ms:
        aptas.append(version)

aptas.sort(key=lambda version: version["rmse_min"])
if aptas:
    print("Versión elegida:", aptas[0]["version"])
else:
    print("Ninguna versión cumple el límite de latencia.")

La versión `v1` cumple el límite del ejemplo. La versión `v2` reduce el error, con una mediana de 210 ms. Cambia el límite a 220 ms y comprueba qué versión elegirías. El experimento ilustra cómo una condición de operación influye en la selección.

## Práctica por equipo

Trabajen con los mismos equipos que se formaron para la actividad de Docker de la Clase 17.

Sigue la guía de tu equipo y copia su starter al repositorio privado. Los archivos ya están escritos. La guía indica qué configurar, ejecutar y comprobar. En drift y SHAP puedes usar el notebook incluido o ejecutar el script desde la terminal.

| Equipo | Guía | Comprobación principal |
|---|---|---|
| 1 | [Rendimiento de la API](../docs/tareas/laboratorios-mlops/01-rendimiento-api.md) | Comparar error y latencia de dos modelos. |
| 2 | [Integración continua](../docs/tareas/laboratorios-mlops/02-integracion-continua.md) | Mostrar una prueba que detecta una regresión y pasa después de corregirla. |
| 3 | [Despliegue continuo en EC2](../docs/tareas/laboratorios-mlops/03-despliegue-continuo.md) | Actualizar el servicio, comprobar la versión y recuperar la anterior. |
| 4 | [Drift y reentrenamiento](../docs/tareas/laboratorios-mlops/04-drift-reentrenamiento.md) | Activar un entrenamiento condicionado y evaluar un candidato. |
| 5 | [Explicabilidad con SHAP](../docs/tareas/laboratorios-mlops/05-explicabilidad-shap.md) | Comparar importancias globales y explicar un viaje particular. |

Consulta la [consigna y rúbrica](../docs/tareas/laboratorios-mlops/README.md) para preparar el repositorio y la evidencia.

## Errores frecuentes

- **La API responde, pero no carga la revisión esperada.** Compara la versión devuelta con la que preparaste para la ejecución.
- **La prueba falla antes de enviar una solicitud.** Revisa el ambiente, la ruta del artefacto y el mensaje de arranque de la API.
- **El detector señala un cambio en las zonas.** Para saber si cambió el error del modelo, compara las estimaciones con las duraciones observadas.
- **La explicación muestra muchas variables de zona.** Comprueba cómo salieron del preprocesamiento y agrúpalas según la característica original.

## Recopilación

Cada laboratorio produce evidencia para una decisión. Las pruebas ayudan a decidir si un cambio se puede integrar. La versión observada confirma qué modelo está en servicio. Las mediciones, los datos recientes y las explicaciones permiten investigar cómo está funcionando y qué conviene revisar.

## Comprobación final

- Ejecuta la comprobación principal de tu guía y explica qué demuestra el resultado.
- Comprueba que el README permite repetirla desde un clon nuevo.
- Revisa que las evidencias aparezcan enlazadas y que el repositorio sea accesible para el profesor.

## Presentación del proyecto final

Propongo dos fechas para la presentación del proyecto. Como grupo, elijan la que prefieran.

- **Lunes 30 de noviembre de 2026.** Presentan el proyecto ese día.
- **Miércoles 2 de diciembre de 2026.** El lunes 30 se dedica a una asesoría para revisar la retroalimentación de la tercera entrega del proyecto y preparar los ajustes antes de presentar.

La fecha se confirma cuando el grupo haya elegido.